# DA-Fusion + ResNet50 — Kaggle Submission

Mirrors `src/shared/resnet_baseline.ipynb` (ResNet50 → BiomassNet MLP, 5-fold CV, hflip augmentation) and adds DA-Fusion synthetic images as extra training data.

**Data leakage prevention**: for each fold, we only include synthetics whose `source_image` is in that fold's training split — no synthetic ever appears in a validation fold.

**Required Kaggle inputs:**
- Competition: `csiro-biomass`
- Dataset: `mayaevensen/resnet50-pretrained-weights` (ResNet50 `.pth`)
- Dataset: `ragnhildklette/da-fusion-synthetic-biomass` (synthetic images + `train_augmented.csv`)

In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

TARGETS       = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHT_VECTOR = np.array([0.1, 0.1, 0.1, 0.2, 0.5], dtype=np.float32)

cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir   = "/kaggle/input/competitions/csiro-biomass",
    weights_path  = "/kaggle/input/datasets/mayaevensen/resnet50-pretrained-weights/resnet50-11ad3fa6.pth",
    synthetic_dir = "/kaggle/input/datasets/ragnhildklette/da-fusion-synthetic-biomass/augmented",
    synthetic_csv = "/kaggle/input/datasets/ragnhildklette/da-fusion-synthetic-biomass/train_augmented.csv",
    output_dir    = "/kaggle/working",
    # --- Training ---
    epochs       = 200,
    batch_size   = 32,
    lr           = 3e-4,
    weight_decay = 1e-3,
    patience     = 30,
    n_folds      = 5,
)

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
print("train.csv:    ", os.path.exists(TRAIN_CSV))
print("test.csv:     ", os.path.exists(TEST_CSV))
print("train img dir:", os.path.exists(TRAIN_IMG_DIR))
print("test  img dir:", os.path.exists(TEST_IMG_DIR))
print("resnet wts:   ", os.path.exists(cfg.weights_path))
print("synth dir:    ", os.path.exists(cfg.synthetic_dir))
print("synth csv:    ", os.path.exists(cfg.synthetic_csv))

In [ ]:
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]
    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target",
    ).reset_index()
    return df_wide, df_wide[TARGETS].values.astype(np.float32)


def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR.astype(np.float64)
    yt, yp = y_true.reshape(-1), y_pred.reshape(-1)
    ww     = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)
    pred_dict = {img_id: dict(zip(TARGETS, row))
                 for img_id, row in zip(image_ids, predictions)}
    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        return max(0.0, float(pred_dict.get(img_id, {}).get(row["target_name"], 0.0)))
    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]


_LOSS_W = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)

def weighted_smooth_l1(pred, target, beta=1.0):
    loss = F.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss * _LOSS_W.to(pred.device)).mean()

In [ ]:
CANONICAL_SIZE = (2000, 1000)  # (W, H) — synthetics (512x512) get resized to match reals before left/right split.

class ResNet50Extractor:
    """
    Loads pretrained ResNet50 from local .pth file.
    Splits each image into left/right halves -> 2x2048 = 4096-d vector.
    Real images are natively 2000x1000; synthetics are resized to the same canonical size first.
    Supports horizontal flip (train-only augmentation for real images).
    """
    PREPROCESS = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, weights_path, device="cpu"):
        self.device = device
        resnet = models.resnet50(weights=None)
        resnet.load_state_dict(torch.load(weights_path, map_location="cpu"))
        self.backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        print("Loaded pretrained ResNet50 weights.")

    @torch.no_grad()
    def _extract(self, crop):
        t = self.PREPROCESS(crop).unsqueeze(0).to(self.device)
        return self.backbone(t).flatten().cpu().numpy().astype(np.float32)

    def get_features(self, image_path, hflip=False, resize_to_canonical=False):
        if not os.path.exists(image_path):
            print("Missing:", image_path)
            return np.zeros(4096, dtype=np.float32)
        try:
            img = Image.open(image_path).convert("RGB")
            if resize_to_canonical:
                img = img.resize(CANONICAL_SIZE, Image.BILINEAR)
            if hflip:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            w, _ = img.size
            mid = w // 2
            left  = img.crop((0,   0, mid, img.size[1]))
            right = img.crop((mid, 0, w,   img.size[1]))
            return np.concatenate([self._extract(left), self._extract(right)])
        except Exception as e:
            print(f"Error {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

In [ ]:
train_df, y_orig = load_train_data(TRAIN_CSV)
n = len(train_df)
print(f"Training images: {n}")

extractor = ResNet50Extractor(cfg.weights_path, device=DEVICE)

X_orig = np.zeros((n, 4096), dtype=np.float32)
X_flip = np.zeros((n, 4096), dtype=np.float32)
real_ids = []

for i, row in enumerate(train_df.itertuples(index=False)):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row.image_id}.jpg")
    X_orig[i] = extractor.get_features(img_path, hflip=False)
    X_flip[i] = extractor.get_features(img_path, hflip=True)
    real_ids.append(row.image_id)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{n}")

real_ids = np.array(real_ids)
print("Real features (orig) :", X_orig.shape)
print("Real features (flip) :", X_flip.shape)

In [ ]:
df_test = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
test_unique = df_test.drop_duplicates("image_id").copy()

X_test, test_ids = [], []
for row in test_unique.itertuples(index=False):
    img_path = os.path.join(TEST_IMG_DIR, f"{row.image_id}.jpg")
    X_test.append(extractor.get_features(img_path, hflip=False))
    test_ids.append(row.image_id)
X_test = np.array(X_test, dtype=np.float32)
print("Test features:", X_test.shape)

In [ ]:
# --- Load DA-Fusion synthetic CSV ---
df_synth = pd.read_csv(cfg.synthetic_csv)
print(f"Synthetic rows in CSV: {len(df_synth)}")
print("Columns:", list(df_synth.columns))


def _source_id_from_path(p):
    """Extract source image_id (e.g. 'ID123') from a source_image path or filename."""
    stem = Path(str(p)).stem
    return stem.split("_")[0] if "_" in stem else stem


def _resolve_synth_path(image_path_field):
    """Try several ways to locate the synthetic image under cfg.synthetic_dir."""
    candidates = [
        os.path.join(cfg.synthetic_dir, image_path_field),
        os.path.join(cfg.synthetic_dir, os.path.basename(image_path_field)),
        os.path.join(cfg.synthetic_dir, Path(image_path_field).name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None


df_synth["source_id"]  = df_synth["source_image"].apply(_source_id_from_path)
df_synth["resolved"]   = df_synth["image_path"].apply(_resolve_synth_path)
missing = df_synth["resolved"].isna().sum()
print(f"Synthetics found on disk: {len(df_synth) - missing}/{len(df_synth)}")

df_synth = df_synth.dropna(subset=["resolved"]).reset_index(drop=True)

# Filter to synthetics whose source is in our training set (sanity: should be all of them).
train_id_set = set(real_ids.tolist())
df_synth = df_synth[df_synth["source_id"].isin(train_id_set)].reset_index(drop=True)
print(f"Synthetics usable after source-id filter: {len(df_synth)}")

In [ ]:
# --- Extract ResNet features for synthetic images (no flip, resized to canonical 2000x1000) ---
ns = len(df_synth)
X_synth      = np.zeros((ns, 4096), dtype=np.float32)
y_synth      = df_synth[TARGETS].values.astype(np.float32)
synth_source = df_synth["source_id"].values

for i, row in enumerate(df_synth.itertuples(index=False)):
    X_synth[i] = extractor.get_features(row.resolved, hflip=False, resize_to_canonical=True)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{ns}")

print("Synthetic features:", X_synth.shape)
print("Synthetic targets :", y_synth.shape)

In [ ]:
class BiomassNet(nn.Module):
    """MLP with BatchNorm + GELU + Dropout (same as Maya's resnet_baseline)."""
    def __init__(self, input_dim=4096, hidden_dims=(512, 256, 128),
                 output_dim=5, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_fold(X_tr, y_tr, X_val, y_val, feat_scaler, tgt_scaler,
               epochs=200, batch_size=32, lr=3e-4, weight_decay=1e-3,
               patience=30, device="cpu"):
    """Train one BiomassNet. Returns (model, best_val_r2, val_preds_in_original_scale)."""
    X_tr_s  = feat_scaler.transform(X_tr).astype(np.float32)
    X_val_s = feat_scaler.transform(X_val).astype(np.float32)
    y_tr_s  = tgt_scaler.transform(y_tr).astype(np.float32)

    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

    tr_loader  = loader(X_tr_s, y_tr_s, shuffle=True)
    val_loader = loader(X_val_s, y_val.astype(np.float32), shuffle=False)

    model = BiomassNet(input_dim=X_tr.shape[1]).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr/100)

    best_r2, best_state, wait = -np.inf, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            weighted_smooth_l1(model(xb), yb).backward()
            opt.step()
        sched.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds.append(model(xb.to(device)).cpu().numpy())
        preds_orig = tgt_scaler.inverse_transform(np.concatenate(preds))
        r2 = weighted_global_r2(y_val, preds_orig)

        if r2 > best_r2:
            best_r2   = r2
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stop at epoch {epoch}  best R2={best_r2:.4f}")
                break

        if epoch == 1 or epoch % 25 == 0:
            print(f"    ep {epoch:3d}  val_R2={r2:.4f}  best={best_r2:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_preds_s = model(torch.tensor(X_val_s).to(device)).cpu().numpy()
    return model, best_r2, tgt_scaler.inverse_transform(final_preds_s)

In [ ]:
# 5-fold CV with DA-Fusion synthetic injection.
# Synthetics are added ONLY if their source_image is in the training fold — prevents leakage.
kf         = KFold(n_splits=cfg.n_folds, shuffle=True, random_state=SEED)
oof_preds  = np.zeros_like(y_orig)
test_preds = np.zeros((len(test_ids), 5))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_orig), 1):
    print(f"\n{'='*55}\n  FOLD {fold}/{cfg.n_folds}\n{'='*55}")

    fold_train_ids = set(real_ids[tr_idx].tolist())
    mask = np.array([sid in fold_train_ids for sid in synth_source])
    X_syn_f = X_synth[mask]
    y_syn_f = y_synth[mask]
    print(f"  Real train: {len(tr_idx)} (+flip), val: {len(val_idx)}, synth added: {len(X_syn_f)}")

    X_tr = np.concatenate([X_orig[tr_idx], X_flip[tr_idx], X_syn_f])
    y_tr = np.concatenate([y_orig[tr_idx], y_orig[tr_idx], y_syn_f])
    X_val, y_val = X_orig[val_idx], y_orig[val_idx]

    feat_scaler = StandardScaler().fit(X_tr)
    tgt_scaler  = StandardScaler().fit(y_tr)

    model, best_r2, val_preds = train_fold(
        X_tr, y_tr, X_val, y_val,
        feat_scaler=feat_scaler, tgt_scaler=tgt_scaler,
        epochs=cfg.epochs, batch_size=cfg.batch_size,
        lr=cfg.lr, weight_decay=cfg.weight_decay,
        patience=cfg.patience, device=DEVICE,
    )
    oof_preds[val_idx] = val_preds
    fold_scores.append(best_r2)
    print(f"  Fold {fold} best R2: {best_r2:.4f}")

    # Average test predictions across folds
    X_test_s = feat_scaler.transform(X_test).astype(np.float32)
    with torch.no_grad():
        tp_s = model(torch.tensor(X_test_s).to(DEVICE)).cpu().numpy()
    test_preds += tgt_scaler.inverse_transform(tp_s) / cfg.n_folds

print(f"\n{'='*55}")
print(f"OOF weighted R2 : {weighted_global_r2(y_orig, oof_preds):.4f}")
print(f"Per-fold R2     : {[round(s, 4) for s in fold_scores]}")
print(f"RMSE per target : {rmse_per_target(y_orig, oof_preds)}")

In [ ]:
test_preds_clipped = np.clip(test_preds, 0, None)  # biomass can't be negative

submission = prepare_submission(TEST_CSV, test_preds_clipped, test_ids)
sub_path = os.path.join(cfg.output_dir, "submission.csv")
submission.to_csv(sub_path, index=False)
print(f"Saved {sub_path} — rows: {len(submission)}")
print(submission.head(10))